# Lab 6.12 &mdash; Capstone: A Guardrailed LangChain Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Assemble the module: tools + agent + executor + a max_iterations guardrail
- Run the agent over a SUITE of different tasks
- Score tasks solved / total; optionally run a REAL LangChain agent

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps use a tiny **LangChain-shaped shim** (same names &amp; shapes as real LangChain) so they run offline &amp; deterministically. Advanced labs end with an **optional** cell that runs the **real** library.

**Reference:** [Module 6 slides &mdash; Choosing a framework](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-06-12"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Capstone: a **guardrailed LangChain-shaped agent** that ties the module together &mdash; `@tool`
tools, `create_react_agent`, an `AgentExecutor` with a **`max_iterations`** guardrail, and only
**allow-listed** tools registered &mdash; run over a **suite** (a compound task, a lookup, an
arithmetic task). The score is **solved / total**. The optional cell swaps in a **real** LangChain
agent (Ollama/Groq) &mdash; the bridge to Day 4.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

# --- LangChain-SHAPED shim: a tool has .name, .description (from the docstring), .args, .invoke() ---
import inspect
class Tool:
    def __init__(self, fn, name=None, description=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = (description or inspect.getdoc(fn) or "").strip()
        self.args = list(inspect.signature(fn).parameters)
    def invoke(self, value):
        return self.fn(**value) if isinstance(value, dict) else self.fn(value)
    def __repr__(self):
        return "Tool(name=%r)" % self.name
def tool(fn):
    """The @tool decorator -- wrap a plain function into a Tool (mirrors langchain_core.tools.tool)."""
    return Tool(fn)

class AIMessage:
    def __init__(self, content): self.content = content
class FakeChatModel:
    """Deterministic stand-in for ChatOllama / ChatGroq: replays a scripted list of replies.
    Real code: from langchain_ollama import ChatOllama; model = ChatOllama(model="llama3.2:1b").
    Like the real thing, .invoke(prompt) returns a message whose text is in .content."""
    def __init__(self, script): self.script = list(script); self.i = 0
    def invoke(self, prompt):
        reply = self.script[min(self.i, len(self.script) - 1)]; self.i += 1
        return AIMessage(reply)

class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

def create_react_agent(model, tools, prompt):
    """Bind model + tools + prompt into a ReAct agent (mirrors langchain.agents.create_react_agent)."""
    return {"model": model, "tools": {t.name: t for t in tools}, "prompt": prompt}
def parse_react(text):
    """Turn the model's ReAct text into ('final', answer) or ('action', name, input)."""
    action = inp = None
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("Final Answer:"): return ("final", s.split(":", 1)[1].strip())
        if s.startswith("Action Input:"): inp = s.split(":", 1)[1].strip()
        elif s.startswith("Action:"):      action = s.split(":", 1)[1].strip()
    return ("action", action, inp)
class AgentExecutor:
    """Runs the loop: ask model -> parse -> run tool -> observe -> repeat, capped by max_iterations
    (mirrors langchain.agents.AgentExecutor). verbose=True prints the ReAct trace."""
    def __init__(self, agent, tools=None, verbose=False, max_iterations=6):
        self.agent = agent
        self.tools = agent["tools"] if tools is None else {t.name: t for t in tools}
        self.model = agent["model"]; self.prompt = agent["prompt"]
        self.verbose = verbose; self.max_iterations = max_iterations
    def invoke(self, inputs):
        scratch, steps = "", []
        for _ in range(self.max_iterations):
            text = self.model.invoke(self.prompt.format(input=inputs["input"], scratchpad=scratch)).content
            if self.verbose: print(text)
            parsed = parse_react(text)
            if parsed[0] == "final":
                return {"output": parsed[1], "intermediate_steps": steps}
            name, arg = parsed[1], parsed[2]
            obs = self.tools[name].invoke(arg) if name in self.tools else ("unknown tool: %s" % name)
            if self.verbose: print("Observation:", obs)
            steps.append((name, arg, obs)); scratch += text + "\nObservation: " + str(obs) + "\n"
        return {"output": None, "intermediate_steps": steps}

@tool
def lookup(key):
    """Look up a known fact by key."""
    facts = {"population of metropolis": "120000", "capital of france": "Paris"}
    return facts.get(key.lower().strip(), "unknown")
@tool
def calculator(expression):
    """Do exact arithmetic on an expression."""
    return safe_calc(expression)
print("allow-listed tools:", lookup.name, "&", calculator.name)

## Your Turn
Complete `make_executor` (register only the allowed tools; set the guardrail) and `evaluate` (score
the suite). Each task has a scripted plan so the run is deterministic.

In [ ]:
SCRIPTS = {
    "twice the population of metropolis": [
        "Thought: look it up.\nAction: lookup\nAction Input: population of metropolis",
        "Thought: double it.\nAction: calculator\nAction Input: 120000*2",
        "Thought: done.\nFinal Answer: 240000"],
    "capital of france": [
        "Thought: look it up.\nAction: lookup\nAction Input: capital of france",
        "Thought: done.\nFinal Answer: Paris"],
    "compute 15*3": [
        "Thought: compute.\nAction: calculator\nAction Input: 15*3",
        "Thought: done.\nFinal Answer: 45"],
}

def make_executor(goal, max_iterations=6):
    model  = FakeChatModel(SCRIPTS[goal])
    prompt = PromptTemplate.from_template("Q: {input}\n{scratchpad}")
    tools  = ___   # TODO: only the ALLOWED tools (lookup, calculator)
    agent  = create_react_agent(model, tools, prompt)
    return AgentExecutor(agent, max_iterations=___)   # TODO: the guardrail (cap the steps)

def solve(goal):
    return make_executor(goal).invoke({"input": goal})["output"]

SUITE = [
    {"goal": "twice the population of metropolis", "answer": "240000"},
    {"goal": "capital of france", "answer": "Paris"},
    {"goal": "compute 15*3", "answer": "45"},
]

def evaluate():
    solved = ___   # TODO: count suite tasks the agent solves correctly
    return solved, len(SUITE)

try:
    for t in SUITE:
        print(t['goal'], '->', solve(t['goal']))
    print('solved:', evaluate())
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("solves the two-step population task", lambda: solve("twice the population of metropolis") == "240000")
expect_true("solves the capital lookup task", lambda: solve("capital of france") == "Paris")
expect_true("solves the arithmetic task", lambda: solve("compute 15*3") == "45")
expect_true("solves the whole suite (3/3)", lambda: evaluate() == (3, 3))
expect_true("only allow-listed tools are registered", lambda: set(make_executor("capital of france").tools) == {"lookup", "calculator"})
expect_true("the max_iterations guardrail is honored", lambda: make_executor("capital of france", max_iterations=1).invoke({"input": "capital of france"})["output"] is None)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run this against the REAL LangChain (not graded)
Swap the scripted model for a REAL LangChain agent (Ollama llama3.2:1b, or Groq) -- the bridge to Day 4. Safe to skip &mdash; it needs `pip install langchain langchain-ollama` (then
`ollama run llama3.2:1b`) or `langchain-groq` with a `GROQ_API_KEY`; external-API tools also need
their own keys. If a package, model or key is missing the cell prints a friendly note and moves on.
**The graded steps above use a deterministic LangChain-shaped shim, so the lab always verifies offline.**

In [ ]:
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:1b")
    print(llm.invoke("In one sentence, what can an AI agent do that a plain chatbot cannot?").content)
    print("That was a REAL local LLM. Bind it to @tool functions with create_react_agent + AgentExecutor")
    print("(exactly the shapes above) for a production agent.")
except Exception as e:
    print("No local LLM available -- skipping (pip install langchain langchain-ollama + `ollama run llama3.2:1b`,")
    print("or langchain-groq with GROQ_API_KEY):", type(e).__name__)
    print("The guardrailed, offline agent above already solved the whole suite.")
    print("Next: Day 4 -- task automation & multi-agent collaboration.")

---
### Key takeaway
You assembled a guardrailed LangChain-shaped agent that solves a suite with tools, an executor and a step cap. Swap the scripted model for a real LLM and it ships -- next, Day 4 puts agents to work.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>